# **Análisis Técnico del modelo original**



## **1. Descripción Técnica del Modelo Original**

### 1.1 Visión general de la arquitectura

```text
                        ─── INPUTS ───
   Static covariates           Past inputs                Future inputs
   (categórica + real,         (covariables observadas    (covariables conocidas
    invariantes en t)           + conocidas pasadas)       en horizonte)
            │                       │                            │
            ▼                       │                            │
   ┌──────────────────┐    ┌────────┴─────────┐         ┌────────┴─────────┐
   │ Static Covariate │    │   Variable       │         │   Variable       │
   │    Encoders      │    │   Selection      │         │   Selection      │
   │  (4 contextos)   │    │  Network (VSN)   │         │  Network (VSN)   │
   └──────────────────┘    └────────┬─────────┘         └────────┬─────────┘
            │ c_s,c_e,c_c,c_h        │                            │
            │   ┌────────────────────┘                            │
            │   ▼                                                  │
            │  ┌────────────────────────────────┐                  │
            │  │  LSTM Encoder (locality)       │                  │
            │  │  con estado inicial = c_h      │                  │
            │  └─────────────┬──────────────────┘                  │
            │                │ (states)                            │
            │                ▼                                     │
            │  ┌────────────────────────────────┐                  │
            │  │  LSTM Decoder (locality)       │◄─────────────────┘
            │  └─────────────┬──────────────────┘
            │                │ (h_t)
            │                ▼
            │  ┌────────────────────────────────┐
            └─►│  Static Enrichment (GRN(c_e))  │
               └─────────────┬──────────────────┘
                             ▼
               ┌────────────────────────────────┐
               │ Multi-head Interpretable Attn  │  ← causal mask
               │   Q,K,V sobre encoder+decoder   │
               └─────────────┬──────────────────┘
                             ▼
               ┌────────────────────────────────┐
               │ Position-wise FFN + Add&Norm  │
               └─────────────┬──────────────────┘
                             ▼
               ┌────────────────────────────────┐
               │ Quantile heads (P10, P50, P90) │
               └─────────────┬──────────────────┘
                             ▼
                  ŷ_{t+1..t+H}  (B, H, |Q|)
```

**Tres tipos de inputs explícitos** y un único output multi-horizonte con
cuantiles. Toda la red comparte un building block común — la **Gated Residual
Network (GRN)** — que aparece dentro de las VSN, dentro de los static
covariate encoders, dentro del static enrichment y en las cabezas finales.

### 1.2 Componentes principales

#### 1.2.1 Variable Selection Networks (VSN)

Las **VSN** son módulos *gating* que determinan, **por timestep**, qué
variables exógenas son relevantes para el pronóstico en ese instante. La
intuición es que en climatología, por ejemplo, la radiación solar pesa más al
mediodía que a medianoche; las VSN aprenden esa dependencia temporal de
relevancia.

Para un conjunto de variables `{x¹_t, x², ..., x^m_t}` en el timestep `t`:

1. Cada variable se proyecta mediante una GRN propia: `h^j_t = GRN_j(x^j_t)`.
2. Un vector concatenado `Ξ_t = [x¹_t || ... || x^m_t]` pasa por una GRN +
   *softmax* para producir **pesos por variable** `v^χ_t ∈ ℝ^m`.
3. La salida combinada es la suma ponderada:
   `ξ_t = Σⱼ v^χ_{t,j} · h^j_t`.

**Aporte de interpretabilidad**: los pesos `v^χ_t` se pueden inspeccionar
post-hoc para reportar **ranking de importancia por variable** (validación
empírica del análisis de Mutual Information del EDA §6).

Existen **tres VSN** en el TFT: una para inputs estáticos, una para inputs
pasados (observados + conocidos pasados) y una para inputs futuros (conocidos
en horizonte).

#### 1.2.2 Gated Residual Networks (GRN)

La **GRN** es el bloque base reutilizado en todo el modelo. Su forma:

```
GRN(a, c=None):
    η₁ = ELU(W₁ a + W_c c + b₁)            # opcional: contexto estático c
    η₂ = W₂ η₁ + b₂
    η₃ = GLU(η₂)                            # gating: σ(W_g η₂) ⊙ (W_z η₂)
    return LayerNorm(a_skip + η₃)           # skip connection desde a
```

**Funciones clave**:

- **Gating (GLU)** decide cuánto del bloque contribuye realmente — útil para
  *bypass* selectivo cuando el módulo no aporta (ej. dataset pequeño donde
  más capacidad sobreajusta).
- **Skip connection** garantiza que la red puede comportarse como identidad
  si el módulo no es informativo (importante para evitar dependencia
  obligatoria de cada bloque).
- **Contexto estático opcional `c`**: la GRN puede ser modulada por un
  vector estático (proveniente de los static covariate encoders), permitiendo
  que metadatos por entidad afecten el procesamiento dinámico.

#### 1.2.3 LSTM Encoder-Decoder

Un **LSTM bidireccional opcional** procesa los inputs pasados (encoder) y un
**LSTM unidireccional** los inputs futuros (decoder). Su rol es capturar
**dependencias locales** — patrones de minutos a horas — antes de que la
atención multi-head capte las dependencias largas.

**Diferencia con un LSTM clásico**: aquí el LSTM **no produce el output
final**. Su salida `φ_t` se enriquece con contexto estático (Static
Enrichment, §3.2.4) y luego pasa por la atención. El LSTM es un módulo
*upstream* dentro del flujo, no la cabeza del modelo.

**Estado inicial inyectado**: el encoder LSTM inicia con `(c_h, c_c)`
provenientes de los static encoders, de modo que la entidad (estación)
modula la dinámica recurrente desde el primer paso.

#### 1.2.4 Static Enrichment Layer

Tras el LSTM, cada timestep `t` se enriquece con **otro** contexto estático
`c_e` distinto del usado para inicializar el LSTM:

```
θ_t = GRN(φ_t, c_e)
```

Esta etapa permite que la entidad (estación) module la representación
temporal en cada paso, sin diluirse en la recurrencia.

**Por qué cuatro contextos estáticos distintos** (`c_s`, `c_e`, `c_c`,
`c_h`): el paper diseña encoders separados para que cada parte del modelo
"vea" los metadatos por entidad de la forma más útil — uno modula la VSN
estática, otro inicializa el LSTM, otro enriquece la representación post-LSTM
y otro modula el último gating. Es una decisión empírica del paper que
mejora la performance medida.

#### 1.2.5 Multi-head Attention sobre la dimensión temporal

La atención multi-head **causal** (con máscara triangular) se aplica sobre
las representaciones enriquecidas `θ_t` para capturar **dependencias largas**
(más allá del alcance del LSTM). El paper introduce una variante
**interpretable**: en lugar de promediar los outputs de los heads
concatenados, **promedia los scores de atención** entre heads — permitiendo
una matriz `α_{i,j}` única e interpretable como "cuánto el paso `i` mira al
paso `j`".

**Aporte de interpretabilidad**: la matriz `α` es directamente reportable.
Para forecasting de temperatura podemos ver, ej., que la predicción del paso
+24 h presta atención fuerte a los pasos -24 y -168 (mismo periodo del día
anterior y misma hora de la semana anterior) — coincidiendo con los lags
relevantes detectados por la ACF en EDA §5.4.

#### 1.2.6 Quantile Loss

La cabeza final no produce un valor puntual sino **un vector de cuantiles**
`(q̂^{0.1}_t, q̂^{0.5}_t, q̂^{0.9}_t)` para cada paso del horizonte. Se
entrena con la loss:

```
QL_q(y, q̂_q) = max( q · (y − q̂_q), (q − 1) · (y − q̂_q) )
loss(y, q̂)   = Σ_q QL_q(y, q̂_q)  promediada sobre batch y horizonte
```

**Por qué quantile y no MSE**:

- MSE minimiza el **error medio cuadrático** y converge al **valor esperado**
  condicional. Para distribuciones asimétricas o con colas pesadas (eventos
  extremos del EDA §2), el valor esperado está sesgado hacia el centro —
  el modelo **sub-predice** las colas.
- Quantile loss converge al **cuantile condicional** exacto. P50 = mediana
  predicha; P10, P90 = bandas. Sin sub-predicción de colas.
- **Bandas P10–P90 ≈ intervalo de confianza del 80 %** sin necesidad de
  bootstrap o ensembles.

### 1.3 Flujo de datos durante una predicción

Para una muestra (estación A001, ventana terminada en `t`):

1. **Inputs estáticos**: `[station_id=0, region="Centro-Oeste", biome="Cerrado",
   koppen_class="Aw", lat=-15.79, lng=-47.93, alt=1159.5]` → encoders generan
   los cuatro vectores de contexto `c_s, c_e, c_c, c_h`.
2. **Inputs pasados** (`t-167..t`): tensor `(168, n_past)` con
   `[temp_c, humidity_pct, pressure_mb, radiation_kj_m2, wind_speed_ms,
   dew_point_c, hour_sin, hour_cos, doy_sin, doy_cos, month_sin, month_cos]`.
3. **Inputs futuros** (`t+1..t+168`): tensor `(168, n_future)` con sólo las
   covariables conocidas a futuro: `[hour_sin, hour_cos, doy_sin, doy_cos,
   month_sin, month_cos]`. Las exógenas observadas no entran al decoder
   (porque no las conocemos a futuro).
4. **VSN sobre pasado** y **VSN sobre futuro** producen las representaciones
   `ξ^{past}_t` y `ξ^{fut}_t` con pesos por variable inspeccionables.
5. **LSTM encoder** procesa `ξ^{past}_{t-167..t}` partiendo de `c_h` →
   produce `(φ_{t-167}, ..., φ_t)`.
6. **LSTM decoder** procesa `ξ^{fut}_{t+1..t+168}` heredando el estado final
   del encoder → produce `(φ_{t+1}, ..., φ_{t+168})`.
7. **Static enrichment**: cada `φ_τ` se modula con `c_e` → `θ_τ = GRN(φ_τ, c_e)`.
8. **Multi-head self-attention** (causal) sobre `(θ_{t-167}, ..., θ_{t+168})`
   → produce `(β_{t+1}, ..., β_{t+168})` para los pasos del horizonte, junto
   con la matriz `α` de pesos.
9. **Position-wise FFN + Add & Norm** sobre cada `β_τ`.
10. **Quantile heads**: `(q̂^{0.1}_τ, q̂^{0.5}_τ, q̂^{0.9}_τ)` por cada paso
    `τ` del horizonte — output final `(168, 3)`.

### 1.4 Supuestos del modelo

| Supuesto | Estado en este proyecto |
|---|---|
| Series con **frecuencia constante** | ✓ INMET es horario estricto, regularizado en `process.py`. |
| Existencia de covariables temporales **conocidas a futuro** | ✓ Las cíclicas (`hour_sin/cos`, `doy_sin/cos`, `month_sin/cos`) son determinísticas y conocidas a cualquier horizonte. |
| Existencia de covariables **estáticas por entidad** | ✓ `lat`, `lng`, `alt`, `region`, `biome`, `koppen_class`, `station_id` — siete features. |
| Suficientes ejemplos por entidad para entrenar embeddings significativos | **Parcial**: ~52 584 horas/estación en train; suficiente. Con la salvedad de A301 y A615 que tienen menos años cubiertos. |
| **Estacionariedad débil** dentro de la ventana de lookback | **Parcial**: el ADF del EDA §5.3 mostró que la serie original no es estacionaria, pero la diferenciación de orden 1 sí; el modelo absorbe la no-estacionariedad vía la normalización por estación + LSTM + atención. |
| **Independencia entre entidades** | **Aproximada**: en realidad las estaciones cercanas correlacionan; el TFT no modela esto explícitamente. Mitigación: el embedding de `region` captura algo de la correlación espacial agregada. |
| **Distribución gaussiana del residuo** | **No**: la quantile loss no asume distribución alguna; supera al MSE en colas asimétricas (eventos extremos). |

## **2. Relación con los Modelos del Benchmark**

### 2.1 Tabla comparativa

In [3]:
relation = pd.DataFrame([
    {"modelo_benchmark": "Persistencia",
     "similitudes_con_TFT": "Ninguna a nivel de aprendizaje; sí comparten el output multi-horizonte (vector de 168).",
     "diferencias_clave": "Persistencia no aprende; TFT aprende embeddings, atención, VSN, quantile loss. Persistencia es el piso obligatorio."},
    {"modelo_benchmark": "LSTM vanilla",
     "similitudes_con_TFT": "Comparten el LSTM como bloque temporal local (encoder).",
     "diferencias_clave": "TFT añade VSN (selección de variables), GRN (gating), multi-head attention (long-range), embeddings de entidad y quantile loss. LSTM solo aplana al final con un Linear."},
    {"modelo_benchmark": "GRU",
     "similitudes_con_TFT": "Familia recurrente; estructura similar al LSTM dentro del TFT.",
     "diferencias_clave": "GRU usa una celda más simple (3 gates → 2 gates); TFT integra LSTM dentro de un sistema mucho más amplio. GRU no tiene atención ni embeddings."},
    {"modelo_benchmark": "N-BEATSx",
     "similitudes_con_TFT": "Multi-horizonte single-shot; ambos descomponen el pronóstico (TFT vía atención + VSN, N-BEATS vía bases trend/seasonality).",
     "diferencias_clave": "N-BEATS no usa atención ni embeddings nativos. TFT modula el pronóstico con metadatos por entidad; N-BEATS no nativamente."},
    {"modelo_benchmark": "Informer",
     "similitudes_con_TFT": "Ambos son arquitecturas con multi-head attention para forecasting de larga secuencia; ambos manejan 168 h.",
     "diferencias_clave": "Informer privilegia eficiencia (ProbSparse) y no tiene VSN ni embeddings nativos. TFT privilegia interpretabilidad y entidades."},
])
relation

,modelo_benchmark,similitudes_con_TFT,diferencias_clave
0,Persistencia,Ninguna a nivel de aprendizaje; sí comparten e...,Persistencia no aprende; TFT aprende embedding...
1,LSTM vanilla,Comparten el LSTM como bloque temporal local (...,"TFT añade VSN (selección de variables), GRN (g..."
2,GRU,Familia recurrente; estructura similar al LSTM...,GRU usa una celda más simple (3 gates → 2 gate...
3,N-BEATSx,Multi-horizonte single-shot; ambos descomponen...,N-BEATS no usa atención ni embeddings nativos....
4,Informer,Ambos son arquitecturas con multi-head attenti...,Informer privilegia eficiencia (ProbSparse) y ...


## **3. Conexión con las Decisiones del EDA**

In [4]:
eda_link = pd.DataFrame([
    {"decision_EDA": "Estacionalidad diaria + anual fuerte (FFT picos 24h, 12h, 8766h; STL ~50–80 % varianza estacional)",
     "como_TFT_la_honra": "Las cíclicas (`hour_sin/cos`, `doy_sin/cos`, `month_sin/cos`) entran como **'known future inputs'** y la VSN futura les asigna pesos automáticamente. El multi-head attention captura los lags 24/168 nativamente."},
    {"decision_EDA": "Heterogeneidad regional dramática (Norte 22-32 °C plano vs Sul 5-35 °C amplio)",
     "como_TFT_la_honra": "**Static covariates** (`region`, `biome`, `koppen_class`, `lat/lng/alt`) se inyectan vía los 4 contextos estáticos, modulando VSN, LSTM, static-enrichment y output."},
    {"decision_EDA": "Lookback de 168 h justificado por ACF/PACF significativa hasta lag 168",
     "como_TFT_la_honra": "**`max_encoder_length=168`** directamente; la atención causal cubre los 168 pasos sin degradación a diferencia de LSTM puro."},
    {"decision_EDA": "Multi-horizonte simultáneo {24, 72, 168}",
     "como_TFT_la_honra": "**`max_prediction_length=168`** con output single-shot; las métricas se reportan en slices h=24, h=72, h=168 sobre el mismo vector de salida."},
    {"decision_EDA": "Eventos extremos en colas (p01/p99) con flag `is_extreme`",
     "como_TFT_la_honra": "**Quantile loss multi-percentil** (P10/P50/P90) modela las colas explícitamente — no sub-predice como MSE. Las bandas P10–P90 ≈ 80 % CI."},
    {"decision_EDA": "~21 % faltantes en `radiation_kj_m2` (gaps nocturnos legítimos)",
     "como_TFT_la_honra": "La **VSN pasada** aprende a darle peso bajo en pasos nocturnos; ventanas con NaN remanente se descartan en el dataset (no se imputan con 0)."},
    {"decision_EDA": "Exógenas seleccionadas: humidity, pressure, radiation, wind_speed, dew_point",
     "como_TFT_la_honra": "Entran como **'past observed inputs'** a la VSN pasada — la VSN reportará empíricamente cuáles son más relevantes (validación del análisis MI del EDA §6)."},
    {"decision_EDA": "Estandarización por estación (scaler fit en train por wmo)",
     "como_TFT_la_honra": "Compatible: la normalización ocurre upstream, el TFT consume los inputs ya estandarizados; la inversión se hace post-hoc al inverse_transform las predicciones."},
    {"decision_EDA": "Anti-leakage temporal estricto (split por años + ventaneo no cruza fronteras)",
     "como_TFT_la_honra": "Compatible: el TFT consume las ventanas que entrega `make_windows`; la causal mask del attention impide adicionalmente cualquier filtración intra-ventana."},
    {"decision_EDA": "Sesgo de representación (Sudeste/Nordeste con más estaciones)",
     "como_TFT_la_honra": "Mitigable con **`WeightedRandomSampler`** ponderado por región — compatible con la implementación oficial. El embedding de `region` además permite al modelo compensar el sesgo internamente."},
])
eda_link

,decision_EDA,como_TFT_la_honra
0,Estacionalidad diaria + anual fuerte (FFT pico...,"Las cíclicas (`hour_sin/cos`, `doy_sin/cos`, `..."
1,Heterogeneidad regional dramática (Norte 22-32...,"**Static covariates** (`region`, `biome`, `kop..."
2,Lookback de 168 h justificado por ACF/PACF sig...,**`max_encoder_length=168`** directamente; la ...
3,"Multi-horizonte simultáneo {24, 72, 168}",**`max_prediction_length=168`** con output sin...
4,Eventos extremos en colas (p01/p99) con flag `...,**Quantile loss multi-percentil** (P10/P50/P90...
5,~21 % faltantes en `radiation_kj_m2` (gaps noc...,La **VSN pasada** aprende a darle peso bajo en...
6,"Exógenas seleccionadas: humidity, pressure, ra...",Entran como **'past observed inputs'** a la VS...
7,Estandarización por estación (scaler fit en tr...,"Compatible: la normalización ocurre upstream, ..."
8,Anti-leakage temporal estricto (split por años...,Compatible: el TFT consume las ventanas que en...
9,Sesgo de representación (Sudeste/Nordeste con ...,Mitigable con **`WeightedRandomSampler`** pond...
